# # Film Eleştirileri ve Bag-of-Words Modellemesi

🎯 Bu zorluğun amacı, metinlerin ***Bag-of-words*** modellemesiyle oynamaktır.

✍️ Aşağıdaki veri setinde, _“olumlu”_ veya _“olumsuz”_ olarak sınıflandırılmış 2000 adet yorum bulunmaktadır.

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv")
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [2]:
data.shape

(2000, 2)

## 1. Ön işleme (Preprocessing)

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek bir `preprocessing` fonksiyonu yazın ve bunu tüm yorumlara uygulayın. Fonksiyon şunları yapmalıdır:
    - boşlukları kaldırma
    - harfleri küçük harfe çevirme
    - sayıları kaldırma
    - noktalama işaretlerini kaldırma
    - tokenization (kelimelere ayırma)
    - lemmatization (kelime köküne indirgeme)
- Temizlenmiş yorumları `clean_reviews` adlı bir sütunda saklayabilirsiniz.
- Bu aşamada stopword’leri kaldırmayın; nedenini `3. N-gram modelleme` bölümünde açıklayacağız.

In [4]:
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
import re



In [11]:
lemmatizer = WordNetLemmatizer()
def preprocessing(sentence):
    
    text = re.sub(r'\d+', '', sentence)
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text) # Removes everything except word characters and spaces
    # Remove spaces (optional, if you want all characters removed)
    text = re.sub(r'\s+', ' ', text).strip()
    text=text.lower()
    text=word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in text]
    # Join the lemmatized words back into a string (optional, you can also keep a list)
    return ' '.join(lemmatized_words)


In [12]:
data["clean_reviews"]=data["reviews"].apply(preprocessing)

In [13]:
data["clean_reviews"]

0       plot two teen couple go to a church party drin...
1       the happy bastard quick movie review damn that...
2       it is movie like these that make a jaded movie...
3       quest for camelot is warner bros first feature...
4       synopsis a mentally unstable man undergoing ps...
                              ...                        
1995    wow what a movie it everything a movie can be ...
1996    richard gere can be a commanding actor but he ...
1997    glorystarring matthew broderick denzel washing...
1998    steven spielberg second epic film on world war...
1999    truman trueman burbank is the perfect name for...
Name: clean_reviews, Length: 2000, dtype: object

❓ **Soru (LabelEncoding)**❓

Hedefinizi LabelEncode ile kodlayın ve `“target_encoded”` adlı bir sütuna kaydedin.

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()

# Fit and transform the column
# .fit_transform() learns the unique categories and transforms the data in one step
data['target_encoded'] = le.fit_transform(data['target'])

In [16]:
for code, label in enumerate(le.classes_):
    print(f"{label}: {code}")

neg: 0
pos: 1


In [17]:
# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


## 2. Bag-of-Words Modellemesi

❓ **Soru (Tek kelimelik sözcüklerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [19]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import CountVectorizer

In [20]:
vectorizer = CountVectorizer()

# Step 3: Fit and transform the text data
# The input to fit_transform should be the Series containing the text
counts = vectorizer.fit_transform(data['clean_reviews'])

# Step 4: Convert the sparse matrix to a dense array and then to a DataFrame
count_array = counts.toarray()
feature_names = vectorizer.get_feature_names_out() # use get_feature_names_out() for modern sklearn

X_bow = pd.DataFrame(data=count_array, columns=feature_names)
mnb = MultinomialNB()

scores = cross_val_score(mnb, X_bow, data["target_encoded"], cv=5, scoring='recall')

In [21]:
scores

array([0.785, 0.81 , 0.79 , 0.83 , 0.8  ])

## 3. N-gram Modellemesi

👀 Stop kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini bigramlarla eğiteceğiz. Bu nedenle, “I do not like coriander” (Kişnişi sevmiyorum) gibi bir cümlede, örneğin bu cümlede olumsuzluğu tespit etmek için “do not” bigramını taramak önemlidir.

❓ **Soru (bigramlarla NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [23]:
from sklearn.model_selection import cross_validate

In [24]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

🏁 Tebrikler! Artık vektörleştirilmiş metinler üzerinde Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

